# Creating Healpix Density Function
We seek to create a map_paritions function capable of finding the object density of the partitions within DES DR2. Density = N * 4^(-order), where order is the healpix order. From there, we will run the pipeline on this high density parition and analyze the results.

In [1]:
import sys
import os

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from pathlib import Path
import importlib

import numpy as np
import pandas as pd
from astropy.io import ascii
import matplotlib.pyplot as plt

from dask.distributed import Client
import dask.array
from dask.dataframe.utils import make_meta

from hats import read_hats
from hats.inspection import plot_pixels
from hats_import.catalog.file_readers import CsvReader
from hats_import.margin_cache.margin_cache_arguments import MarginCacheArguments
from hats_import.pipeline import ImportArguments, pipeline_with_client

import lsdb

from catalog_filtering import bandFilterLenient, contains_PM
import hpms_pipeline_optimized as hpms

print("Imported libraries.")

Imported libraries.


In [11]:
bandList = ['G','R','I','Z','Y']
class_star = None
spread_model = 0.05
magnitude_error = 0.05
check_flags = True
mag = 19
query_string = bandFilterLenient(bandList,classStar=class_star,spreadModel=spread_model,magError=magnitude_error,flag=check_flags,mag=mag)
des_cols = (
    [f'CLASS_STAR_{band}' for band in bandList] + 
    [f'FLAGS_{band}' for band in bandList] + 
    ['RA','DEC','COADD_OBJECT_ID'] + 
    [f'SPREAD_MODEL_{band}' for band in bandList] + 
    [f'WAVG_MAG_PSF_{band}' for band in bandList] + 
    [f'WAVG_MAGERR_PSF_{band}' for band in bandList]
)
k = 2
max_obj_deviation = 0.2
des_id_col = 'COADD_OBJECT_ID_1'
mag_cols = [f'WAVG_MAG_PSF_{band}' for band in ['I']]
min_neighbors = 4
max_neighbor_dist = 24
xmatch_max_neighbors = 100
print("Defined globals.")

Defined globals.


In [12]:
CATALOG_DIR = Path("../../../../catalogs")
MARGIN_CACHE_DIR = CATALOG_DIR / 'margin_caches'

DES_NAME = "des_light"
DES_DIR = CATALOG_DIR / DES_NAME 

DES_MARGIN_CACHE_NAME = "des_margin_cache_18_arcsec"
DES_MARGIN_CACHE_DIR = MARGIN_CACHE_DIR / DES_MARGIN_CACHE_NAME

In [13]:
def subset_len(df, pixel):
    return pd.DataFrame({
        "density": len(df) * 4.0 ** (pixel[0]),
        "pixel_order": pixel[0],
        "pixel_index": pixel[1]
    }, index=[0])

In [14]:
des_dr2 = lsdb.read_hats(DES_DIR)
des_dr2

,CLASS_STAR_G,CLASS_STAR_R,CLASS_STAR_I,CLASS_STAR_Z,CLASS_STAR_Y,FLAGS_G,FLAGS_R,FLAGS_I,FLAGS_Z,FLAGS_Y,RA,DEC,COADD_OBJECT_ID,SPREAD_MODEL_G,SPREAD_MODEL_R,SPREAD_MODEL_I,SPREAD_MODEL_Z,SPREAD_MODEL_Y,WAVG_MAG_PSF_G,WAVG_MAG_PSF_R,WAVG_MAG_PSF_I,WAVG_MAG_PSF_Z,WAVG_MAG_PSF_Y,WAVG_MAGERR_PSF_G,WAVG_MAGERR_PSF_R,WAVG_MAGERR_PSF_I,WAVG_MAGERR_PSF_Z,WAVG_MAGERR_PSF_Y,NEPOCHS_G,NEPOCHS_R,NEPOCHS_I,NEPOCHS_Z,NEPOCHS_Y
npartitions=1582,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 4, Pixel: 0",double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow]
"Order: 5, Pixel: 8",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 743",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 1, Pixel: 47",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [6]:
client = Client()
with client:
    display(client)
    find_lens = des_dr2.map_partitions(subset_len, include_pixel=True).compute()
    
find_lens

/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45009 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:45009/status,
Dashboard: http://127.0.0.1:45009/status,Workers: 16
Total threads: 128,Total memory: 234.38 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41679,Workers: 16
Dashboard: http://127.0.0.1:45009/status,Total threads: 128
Started: Just now,Total memory: 234.38 GiB
Comm: tcp://127.0.0.1:32969,Total threads: 8
Dashboard: http://127.0.0.1:46809/status,Memory: 14.65 GiB
Nanny: tcp://127.0.0.1:46365,


2025-10-04 20:50:16,403 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 20:50:16,506 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 20:50:16,525 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 20:50:16,525 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 20:50:16,526 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 20:50:16,526 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 20:50:16,527 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 20:50:16,527 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 20:50:16,528 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04

,density,pixel_order,pixel_index
0,214766080.0,4,0
0,418678784.0,5,8
...,...,...,...
0,187264.0,3,743
0,1159464.0,1,47


In [9]:
densities = find_lens.reset_index(drop=True)
densities

,density,pixel_order,pixel_index
0,214766080.0,4,0
1,418678784.0,5,8
...,...,...,...
1580,187264.0,3,743
1581,1159464.0,1,47


In [10]:
max_density = densities.loc[densities['density'].idxmax()]
max_density

density        939134976.0
pixel_order            5.0
pixel_index         8456.0
Name: 531, dtype: float64

In [17]:
repart_name = "des_dr2_pix_thresh_100k"
REPART_DIR = CATALOG_DIR / repart_name / repart_name

In [19]:
repart = lsdb.read_hats(REPART_DIR)
repart

,CLASS_STAR_G,CLASS_STAR_R,CLASS_STAR_I,CLASS_STAR_Z,CLASS_STAR_Y,FLAGS_G,FLAGS_R,FLAGS_I,FLAGS_Z,FLAGS_Y,RA,DEC,COADD_OBJECT_ID,SPREAD_MODEL_G,SPREAD_MODEL_R,SPREAD_MODEL_I,SPREAD_MODEL_Z,SPREAD_MODEL_Y,WAVG_MAG_PSF_G,WAVG_MAG_PSF_R,WAVG_MAG_PSF_I,WAVG_MAG_PSF_Z,WAVG_MAG_PSF_Y,WAVG_MAGERR_PSF_G,WAVG_MAGERR_PSF_R,WAVG_MAGERR_PSF_I,WAVG_MAGERR_PSF_Z,WAVG_MAGERR_PSF_Y,NEPOCHS_G,NEPOCHS_R,NEPOCHS_I,NEPOCHS_Z,NEPOCHS_Y
npartitions=23044,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 6, Pixel: 0",double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow]
"Order: 7, Pixel: 6",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 6, Pixel: 48136",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 767",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [20]:
client = Client()
with client:
    display(client)
    find_lens = repart.map_partitions(subset_len, include_pixel=True).compute()
    
find_lens

/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43583 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:43583/status,
Dashboard: http://127.0.0.1:43583/status,Workers: 16
Total threads: 128,Total memory: 234.38 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42611,Workers: 16
Dashboard: http://127.0.0.1:43583/status,Total threads: 128
Started: Just now,Total memory: 234.38 GiB
Comm: tcp://127.0.0.1:37527,Total threads: 8
Dashboard: http://127.0.0.1:35089/status,Memory: 14.65 GiB
Nanny: tcp://127.0.0.1:33497,


/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/distributed/client.py:3383: UserWarning: Sending large graph of size 15.44 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
2025-10-04 21:10:13,478 - tornado.application - ERROR - Uncaught exception GET /status/ws (10.8.11.32)
HTTPServerRequest(protocol='http', host='localhost:43583', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='10.8.11.32')
Traceback (most recent call last):
  File "/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/tornado/web.py", line 1848, in _execute
    result = await result
             ^^^^^^^^^^^^
  File "/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packa

KeyboardInterrupt: 

2025-10-04 21:57:42,197 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:34245'.
2025-10-04 21:57:42,214 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:46275' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('read_pixel-fused-nestedframe-46ab3037bd0dd8684d525924f0e5c580', 2600), ('read_pixel-fused-nestedframe-46ab3037bd0dd8684d525924f0e5c580', 6145), ('read_pixel-fused-nestedframe-46ab3037bd0dd8684d525924f0e5c580', 20783), ('read_pixel-fused-nestedframe-46ab3037bd0dd8684d525924f0e5c580', 394), ('read_pixel-fused-nestedframe-46ab3037bd0dd8684d525924f0e5c580', 3521), ('read_pixel-fused-nestedframe-46ab3037bd0dd8684d525924f0e5c580', 19480), ('read_pixel-fused-nestedframe-46ab3037bd0dd8684d525924f0e5c580', 6639), ('read_pixel-fused-nestedframe-46ab3037bd0dd8684d525924f0e5c580', 1971), ('read_pixel-fused-nestedframe-46ab3037bd0dd8684d525924f0e5c580', 18824), ('read_pixel-fused-nes

#### We see that pixel 8456 (index 8456) is the highest density healpix pixel for DES. We will use this to run memory performance testing. We will use Memray for memory analysis.

In [6]:
import memray

In [7]:
hd_order = 5
hd_idx = 8456
high_density_parition = des_dr2.pixel_search((hd_order, hd_idx))

In [8]:
%load_ext memray

In [9]:
%%memray_flamegraph --native --follow-fork


# # memray kwargs to consider: --follow-fork, --leaks, --temporary-allocations, --temporal, --split-threads, --max-memory-records

hpms.execute_pipeline(high_density_parition, query_string, xmatch_max_neighbors,
                      max_neighbor_dist, min_neighbors, k,
                      max_obj_deviation, des_id_col, mag_cols).to_hats(catalog_name='memray_runs_1', 
                                                                       base_catalog_path=Path("./test_run_cat"))

/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/lsdb/dask/crossmatch_catalog_data.py:147: RuntimeWarning: Right catalog does not have a margin cache. Results may be incomplete and/or inaccurate.
  warnings.warn(


KeyboardInterrupt: 

Output()

Output()

Results saved to /ocean/projects/phy210048p/jpassos/astrophysics/Jupyter 
Notebooks/kth_star_pipeline/testing/memray-results/tmps6mthg3_/flamegraph.html